# Advanced Soccer Visualizations with mplsoccer (EXTRA)

**Chapter 3: Exploratory Data Analysis in Soccer**

## Learning Objectives

By the end of this notebook, you will be able to:
- Install and use the mplsoccer library
- Create professional shot maps
- Build pass location heatmaps
- Customize pitch appearance and styling
- Create publication-quality soccer visualizations

## Prerequisites

- Completed core notebooks 01-04
- Understanding of event-level data
- Basic matplotlib knowledge

## Installation

First, install mplsoccer:
```bash
pip install mplsoccer
```

## Setup and Imports

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from mplsoccer import Pitch, VerticalPitch

# Set style
sns.set_style("white")
plt.rcParams['figure.figsize'] = (12, 8)

## Load Event Data

In [ ]:
# Load events from a match
BASE = Path("open-data/data")
match_id = 22921

events = json.load(open(BASE / f"events/{match_id}.json", "r"))
events_df = pd.json_normalize(events, sep="_")
events_df["match_id"] = match_id

print(f"Loaded {len(events_df)} events")

## Creating a Basic Shot Map

A shot map shows where shots were taken from and whether they resulted in goals. This is one of the most common soccer visualizations.

In [ ]:
# Extract shots
shots = events_df.query("type_name == 'Shot'").copy()
shots["is_goal"] = shots["shot_outcome_name"].eq("Goal")

# Extract coordinates (handle different column naming conventions)
if "location" in shots.columns:
    shots[["x", "y"]] = shots["location"].apply(pd.Series)
elif "location_0" in shots.columns:
    shots["x"] = shots["location_0"]
    shots["y"] = shots["location_1"]

print(f"Total shots: {len(shots)}")
print(f"Goals: {shots['is_goal'].sum()}")

In [ ]:
# Create a horizontal pitch
pitch = Pitch(pitch_color='grass', line_color='white', stripe=True)
fig, ax = pitch.draw(figsize=(12, 8))

# Plot goals and misses with different colors
goals = shots[shots["is_goal"]]
misses = shots[~shots["is_goal"]]

pitch.scatter(goals["x"], goals["y"], 
              color='blue', s=200, edgecolors='white', linewidth=2,
              ax=ax, label='Goal', zorder=3)
pitch.scatter(misses["x"], misses["y"], 
              color='red', s=100, alpha=0.6, edgecolors='white', linewidth=1,
              ax=ax, label='Miss', zorder=2)

ax.legend(loc='upper left', fontsize=12)
plt.title("Shot Map: Goals vs Misses", fontsize=16, pad=20)
plt.tight_layout()
plt.show()

**What this tells us:**
- Blue circles show successful shots (goals)
- Red circles show unsuccessful attempts
- Larger blue circles make goals stand out
- Position on pitch shows shot location
- Clusters near goal indicate high-quality chances

## Shot Map with xG Sizing

We can make the visualization more informative by sizing markers based on expected goals (xG).

In [ ]:
# Check if xG data is available
xg_col = "shot_statsbomb_xg" if "shot_statsbomb_xg" in shots.columns else None

if xg_col:
    # Create pitch
    pitch = Pitch(pitch_color='#22312b', line_color='#efefef', linewidth=2)
    fig, ax = pitch.draw(figsize=(12, 8))
    
    # Size markers by xG (scale up for visibility)
    size_scale = 1000
    
    pitch.scatter(goals["x"], goals["y"], 
                  s=goals[xg_col] * size_scale,
                  color='#00ff85', edgecolors='white', linewidth=2,
                  ax=ax, label='Goal', alpha=0.9, zorder=3)
    pitch.scatter(misses["x"], misses["y"], 
                  s=misses[xg_col] * size_scale,
                  color='#ff4e50', edgecolors='white', linewidth=1,
                  ax=ax, label='Miss', alpha=0.6, zorder=2)
    
    ax.legend(loc='upper left', fontsize=12)
    plt.title("Shot Map with xG Sizing", fontsize=16, pad=20, color='white')
    plt.tight_layout()
    plt.show()
else:
    print("xG data not available in this dataset")

**What this tells us:**
- Larger circles = higher xG (better quality chances)
- Smaller circles = lower xG (difficult shots)
- Green circles show which high-quality chances were converted
- Red circles show missed opportunities
- Helps identify clinical finishing vs wasteful shooting

## Pass Location Heatmap

Heatmaps show where on the pitch events occur most frequently. Let's create a pass start location heatmap for one team.

In [ ]:
# Extract passes for one team
team_name = "France Women's"
passes = events_df.query("type_name == 'Pass' and team_name == @team_name").copy()

# Extract coordinates
if "location" in passes.columns:
    passes[["x", "y"]] = passes["location"].apply(pd.Series)
elif "location_0" in passes.columns:
    passes["x"] = passes["location_0"]
    passes["y"] = passes["location_1"]

# Remove any missing coordinates
passes = passes.dropna(subset=["x", "y"])

print(f"{team_name} made {len(passes)} passes")

In [ ]:
# Create hexbin heatmap
pitch = Pitch(pitch_color='#22312b', line_color='#efefef', linewidth=2)
fig, ax = pitch.draw(figsize=(12, 8))

# Create hexbin plot
hexmap = pitch.hexbin(passes["x"], passes["y"], 
                      ax=ax, edgecolors='#22312b',
                      gridsize=25, cmap='Blues', mincnt=1)

# Add colorbar
cbar = plt.colorbar(hexmap, ax=ax, pad=0.03)
cbar.set_label('Number of Passes', rotation=270, labelpad=20, color='white')
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='white')

plt.title(f"Pass Start Locations — {team_name}", fontsize=16, pad=20, color='white')
plt.tight_layout()
plt.show()

**What this tells us:**
- Darker hexagons = more passes started from that area
- Shows spatial distribution of passing activity
- Reveals team's buildup patterns
- Concentration in defensive third suggests building from the back
- Wide distribution indicates use of full pitch width

## Advanced Heatmap with KDE

Kernel Density Estimation (KDE) creates smooth, continuous heatmaps instead of hexagonal bins.

In [ ]:
# Create pitch with custom styling
pitch = Pitch(pitch_color='#22312b', line_color='#efefef', linewidth=2)
fig, ax = pitch.draw(figsize=(12, 8))

# Create KDE heatmap
pitch.kdeplot(passes["x"], passes["y"], ax=ax, 
              shade=True, levels=10, alpha=0.7, 
              cmap='YlOrRd', zorder=1)

plt.title(f"Pass Density Heatmap — {team_name}", fontsize=16, pad=20, color='white')
plt.tight_layout()
plt.show()

## Vertical Pitch Orientation

Sometimes a vertical pitch layout works better for presentations or publications.

In [ ]:
# Create vertical pitch
pitch = VerticalPitch(pitch_color='grass', line_color='white', stripe=True)
fig, ax = pitch.draw(figsize=(8, 12))

# Plot shots (need to swap x and y for vertical orientation)
pitch.scatter(shots["x"], shots["y"], 
              c=shots["is_goal"].map({True: 'blue', False: 'red'}),
              s=100, edgecolors='white', linewidth=1,
              ax=ax, zorder=2)

plt.title("Vertical Shot Map", fontsize=16, pad=20)
plt.tight_layout()
plt.show()

## Summary

In this notebook, we learned how to:

1. **Install and use mplsoccer** for soccer-specific visualizations
2. **Create shot maps** with goals and misses clearly distinguished
3. **Size markers by xG** to show shot quality
4. **Build pass heatmaps** using hexbins and KDE
5. **Customize pitch appearance** with colors, styles, and orientations
6. **Create publication-quality graphics** suitable for reports and presentations

These visualizations bridge the gap between raw data and tactical understanding. They make patterns immediately visible to coaches, analysts, and fans.

## Next Steps

- Explore passing networks (extra notebook 06)
- Apply these visualizations to the Japan case study (extra notebook 07)
- Experiment with different color schemes and styles
- Create multi-panel figures combining different visualization types

## Practice Exercises

1. **Defensive Actions Map**: Create a pitch map showing tackles, interceptions, and clearances
2. **Comparison Shot Maps**: Create side-by-side shot maps for two teams in the same match
3. **Pass End Locations**: Create a heatmap of where passes ended (not just where they started)
4. **Custom Styling**: Experiment with different color schemes to match team colors
5. **Animated Heatmap**: Create heatmaps for different time periods of the match (0-30, 30-60, 60-90 minutes)